# H-1B Visa Data Analysis: Data Cleaning Pipeline

This notebook cleans and reshapes raw H-1B petition data from the USCIS H-1B Employer Data Hub into a tidy format that powers an interactive Tableau dashboard.

**Source:** [USCIS H-1B Employer Data Hub](https://www.uscis.gov/tools/reports-and-studies/h-1b-employer-data-hub), one file per fiscal year, 2015 through 2025.

**Output:** `h1b_cleaned.csv`, one row per employer, industry, city, state, fiscal year, and petition type, with approvals, denials, total petitions, and approval rate.

**Approach:** Append the yearly files, clean and standardize columns, remove incomplete records, reshape from wide (12 approval/denial columns) to long (petition type as a dimension), add calculated fields, and validate that petition totals are preserved end to end.

## 1. Load and combine raw files

Read all 11 yearly files and append them into a single dataframe. USCIS crosstab exports are UTF-16 encoded and tab-separated.

In [10]:
!pip install pandas



Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import pandas as pd
import glob

# Load all raw yearly files from the USCIS H-1B Employer Data Hub
files = sorted(glob.glob('Employer Information_*.csv'))

# USCIS crosstab exports are UTF-16 encoded and tab-separated
df_list = [pd.read_csv(f, encoding='utf-16', sep='\t', low_memory=False) for f in files]
df = pd.concat(df_list, ignore_index=True)

print(f"Loaded {len(files)} files")
print(f"Combined shape: {df.shape}")

Loaded 11 files
Combined shape: (637888, 20)


In [12]:
print(df.shape)

(637888, 20)


## 2. Clean columns

Convert the count columns to numeric (the source has mixed types), drop fields not needed for the dashboard (row ID, Tax ID, zip code), and rename the remaining columns to readable names.

In [13]:
# Convert the 12 count columns to numeric (source has mixed types)
count_cols_raw = [c for c in df.columns if 'Approval' in c or 'Denial' in c]
df[count_cols_raw] = df[count_cols_raw].apply(pd.to_numeric, errors='coerce')

# Steps 3 & 4: Drop unneeded columns and rename the rest
df = df.drop(columns=['Line by line', 'Tax ID', 'Petitioner Zip Code'])
df = df.rename(columns={
    'Fiscal Year   ': 'Fiscal Year',
    'Employer (Petitioner) Name': 'Employer',
    'Industry (NAICS) Code': 'Industry',
    'Petitioner City': 'City',
    'Petitioner State': 'State'
})

print(f"Shape after drop/rename: {df.shape}")
print(list(df.columns))

Shape after drop/rename: (637888, 17)
['Fiscal Year', 'Employer', 'Industry', 'City', 'State', 'New Employment Approval', 'New Employment Denial', 'Continuation Approval', 'Continuation Denial', 'Change with Same Employer Approval', 'Change with Same Employer Denial', 'New Concurrent Approval', 'New Concurrent Denial', 'Change of Employer Approval', 'Change of Employer Denial', 'Amended Approval', 'Amended Denial']


## 3. Handle missing values and regroup

Remove rows with missing fields, since a drill-down dashboard cannot meaningfully filter on incomplete records. Then regroup by the reference variables: dropping Tax ID and zip can leave near-duplicate rows that must be recombined and summed.

In [14]:
# Step 5: Remove rows with missing values
rows_before = len(df)
df = df.dropna()
print(f"Rows removed (missing values): {rows_before - len(df):,}")

# Step 6: Regroup by reference variables and sum counts
# (dropping Tax ID and Zip can leave near-duplicate rows that must be recombined)
reference_cols = ['Fiscal Year', 'Employer', 'Industry', 'City', 'State']
count_cols = [c for c in df.columns if 'Approval' in c or 'Denial' in c]
df = df.groupby(reference_cols, as_index=False)[count_cols].sum()

# Capture baseline totals immediately before reshape (for integrity check)
approval_cols = [c for c in df.columns if 'Approval' in c]
denial_cols = [c for c in df.columns if 'Denial' in c]
approvals_baseline = df[approval_cols].sum().sum()
denials_baseline = df[denial_cols].sum().sum()

print(f"Shape after regroup: {df.shape}")
print(f"Baseline approvals: {approvals_baseline:,.0f}")
print(f"Baseline denials:   {denials_baseline:,.0f}")

Rows removed (missing values): 37,465
Shape after regroup: (577919, 17)
Baseline approvals: 3,220,767
Baseline denials:   213,140


## 4. Reshape from wide to long

Transform the 12 approval/denial columns into a tidy format with **Petition Type** as a dimension and separate **Approvals** and **Denials** columns. This is built by stacking each petition type's columns directly rather than melting and pivoting, which is far more memory-efficient on the full dataset. The integrity check confirms the reshape preserved every approval and denial.

In [15]:
# Steps 7-10: Reshape from wide to long using a lighter, memory-safe approach

# Build a long frame directly: one row per reference combo + petition type,
# with Approvals and Denials as columns
petition_types = ['New Employment', 'Continuation', 'Change with Same Employer',
                  'New Concurrent', 'Change of Employer', 'Amended']

frames = []
for pt in petition_types:
    part = df[reference_cols].copy()
    part['Petition Type'] = pt
    part['Approvals'] = df[f'{pt} Approval'].values
    part['Denials'] = df[f'{pt} Denial'].values
    frames.append(part)

reshaped = pd.concat(frames, ignore_index=True)

# Integrity check: totals must match the pre-reshape baseline exactly
print(f"Shape after reshape: {reshaped.shape}")
print(f"Approvals: {reshaped['Approvals'].sum():,.0f}  (baseline: {approvals_baseline:,.0f})")
print(f"Denials:   {reshaped['Denials'].sum():,.0f}  (baseline: {denials_baseline:,.0f})")

Shape after reshape: (3467514, 8)
Approvals: 3,220,767  (baseline: 3,220,767)
Denials:   213,140  (baseline: 213,140)


## 5. Calculated fields and final cleanup

Add **Petitions** (approvals plus denials) and **Approval %**, strip the NAICS code prefix from industry names, and remove the zero-petition rows the reshape created for petition types an employer never used.

In [16]:
# Step 11: Calculated columns
reshaped['Petitions'] = reshaped['Approvals'] + reshaped['Denials']
reshaped['Approval %'] = reshaped['Approvals'] / reshaped['Petitions']

# Step 12: Remove NAICS code prefix from Industry (e.g. "51 - Information" -> "Information")
reshaped['Industry'] = reshaped['Industry'].str.replace(r'^\d+\s*-\s*', '', regex=True)

# Step 13: Remove rows with zero petitions (reshape created rows for unused petition types)
rows_before = len(reshaped)
reshaped = reshaped[reshaped['Petitions'] > 0].copy()
print(f"Zero-petition rows removed: {rows_before - len(reshaped):,}")
print(f"Final shape: {reshaped.shape}")

Zero-petition rows removed: 2,560,246
Final shape: (907268, 10)


## 6. Validate and export

Confirm the final approval and denial totals match the pre-reshape baseline exactly, proving no data was lost or duplicated through the entire pipeline. Then export the cleaned dataset.

In [17]:
# Final integrity check: approvals and denials must still match the original baseline
print(f"Final approvals: {reshaped['Approvals'].sum():,.0f}  (baseline: {approvals_baseline:,.0f})")
print(f"Final denials:   {reshaped['Denials'].sum():,.0f}  (baseline: {denials_baseline:,.0f})")

# Step 14: Export the cleaned dataset
reshaped.to_csv('h1b_cleaned.csv', index=False)
print("\nExported to h1b_cleaned.csv")
reshaped.head()

Final approvals: 3,220,767  (baseline: 3,220,767)
Final denials:   213,140  (baseline: 213,140)

Exported to h1b_cleaned.csv


,Fiscal Year,Employer,Industry,City,State,Petition Type,Approvals,Denials,Petitions,Approval %
0,2015,& TV COMMUNICATIONS INC,"Professional, Scientific, and Technical Services",LOS ANGELES,CA,New Employment,1.0,0.0,1.0,1.0
1,2015,& TV COMMUNICATIONS INC,Administrative and Support and Waste Managemen...,LOS ANGELES,CA,New Employment,1.0,0.0,1.0,1.0
3,2015,' BEAM LLC DBA AMERICANA GAME S,Information,SAN FRANCISCO,CA,New Employment,1.0,0.0,1.0,1.0
6,2015,0956588 BC LTD DBA PROCOGIA,"Professional, Scientific, and Technical Services",ISSAQUAH,WA,New Employment,1.0,0.0,1.0,1.0
8,2015,1 HOTEL SOUTH BEACH INC,Accommodation and Food Services,MIAMI BEACH,FL,New Employment,1.0,0.0,1.0,1.0


## 7. State lookup table

A small helper table mapping state abbreviations to full state names, joined in Tableau for clean map tooltips. This is a standalone reference table, separate from the main cleaning pipeline.

In [18]:
# State lookup table: maps state abbreviations to full names for map tooltips in Tableau
state_lookup = {
    'AL':'Alabama','AK':'Alaska','AZ':'Arizona','AR':'Arkansas','CA':'California',
    'CO':'Colorado','CT':'Connecticut','DE':'Delaware','FL':'Florida','GA':'Georgia',
    'HI':'Hawaii','ID':'Idaho','IL':'Illinois','IN':'Indiana','IA':'Iowa',
    'KS':'Kansas','KY':'Kentucky','LA':'Louisiana','ME':'Maine','MD':'Maryland',
    'MA':'Massachusetts','MI':'Michigan','MN':'Minnesota','MS':'Mississippi','MO':'Missouri',
    'MT':'Montana','NE':'Nebraska','NV':'Nevada','NH':'New Hampshire','NJ':'New Jersey',
    'NM':'New Mexico','NY':'New York','NC':'North Carolina','ND':'North Dakota','OH':'Ohio',
    'OK':'Oklahoma','OR':'Oregon','PA':'Pennsylvania','RI':'Rhode Island','SC':'South Carolina',
    'SD':'South Dakota','TN':'Tennessee','TX':'Texas','UT':'Utah','VT':'Vermont',
    'VA':'Virginia','WA':'Washington','WV':'West Virginia','WI':'Wisconsin','WY':'Wyoming',
    'DC':'District of Columbia'
}

state_df = pd.DataFrame(list(state_lookup.items()), columns=['State', 'State Name'])
state_df.to_csv('state_lookup.csv', index=False)
print(f"State lookup created: {state_df.shape[0]} states")
state_df.head()

State lookup created: 51 states


,State,State Name
0,AL,Alabama
1,AK,Alaska
2,AZ,Arizona
3,AR,Arkansas
4,CA,California
